# Part 4 — Mitigation: Making the Classifier Fairer and More Robust
**Assignment 2 | Responsible & Explainable AI | FAST-NUCES**

Three bias mitigation techniques covering pre-processing, post-processing, and data-level intervention:

| Technique | Stage | Method |
|-----------|-------|--------|
| 1. Reweighing | Pre-processing | AIF360 `Reweighing` — per-sample weights in training |
| 2. Threshold Optimization | Post-processing | fairlearn `ThresholdOptimizer` (equalized_odds) |
| 3. Oversampling | Data-level | Duplicate high-black cohort 3× in training set |

**Goal**: Reduce FPR disparity and Statistical Parity Difference while preserving overall F1.

In [1]:
#!pip install transformers torch scikit-learn fairlearn aif360 pandas numpy matplotlib seaborn -q

In [2]:
import pandas as pd
import numpy as np
import torch
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, f1_score, confusion_matrix, ConfusionMatrixDisplay,
    precision_score, recall_score
)
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer
)
from torch.utils.data import Dataset as TorchDataset

SEED = 42
np.random.seed(SEED); torch.manual_seed(SEED)
OPERATING_THRESHOLD = 0.4
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
MODEL_NAME = 'distilbert-base-uncased'
print(f'Device: {DEVICE}')

Device: cuda


In [3]:
# ── Shared utilities ─────────────────────────────────────────────────────────
class ToxicityDataset(TorchDataset):
    def __init__(self, encodings, labels, weights=None):
        self.encodings = encodings
        self.labels    = labels
        self.weights   = weights  # optional per-sample weights
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        if self.weights is not None:
            item['sample_weight'] = torch.tensor(self.weights[idx], dtype=torch.float)
        return item

class WeightedTrainer(Trainer):
    """Trainer that uses per-sample weights from the dataset (for Reweighing)."""
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        weights = inputs.pop('sample_weight', None)
        labels  = inputs.pop('labels')
        outputs = model(**inputs)
        logits  = outputs.logits
        loss_fn = torch.nn.CrossEntropyLoss(reduction='none')
        loss = loss_fn(logits, labels)
        if weights is not None:
            loss = (loss * weights).mean()
        else:
            loss = loss.mean()
        return (loss, outputs) if return_outputs else loss

def tokenize(texts, tok):
    return tok(texts, max_length=128, truncation=True, padding='max_length', return_tensors=None)

def get_probs(model, tok, texts, batch_size=64):
    model.eval()
    all_probs = []
    for i in range(0, len(texts), batch_size):
        batch = list(texts[i:i+batch_size])
        enc = tok(batch, max_length=128, truncation=True, padding=True, return_tensors='pt').to(DEVICE)
        with torch.no_grad():
            probs = torch.softmax(model(**enc).logits, dim=-1).cpu().numpy()[:, 1]
        all_probs.append(probs)
    return np.concatenate(all_probs)

def compute_metrics_fn(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {'accuracy': accuracy_score(labels, preds),
            'f1_macro': f1_score(labels, preds, average='macro')}

def cohort_stats(y_true, probs, thresh=OPERATING_THRESHOLD):
    y_pred = (probs >= thresh).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0,1]).ravel()
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
    fnr = fn / (fn + tp) if (fn + tp) > 0 else 0
    tpr = tp / (tp + fn) if (tp + fn) > 0 else 0
    return dict(FPR=fpr, FNR=fnr, TPR=tpr,
                F1=f1_score(y_true, y_pred, average='macro'),
                PosRate=y_pred.mean())

print('Utilities defined.')

Utilities defined.


In [4]:
# ── Load data and baseline metrics ───────────────────────────────────────────
train_df = pd.read_csv('train_subset.csv')
eval_df  = pd.read_csv('eval_subset.csv')

for col in ['black', 'white']:
    train_df[col] = train_df[col].fillna(0.0)
    eval_df[col]  = eval_df[col].fillna(0.0)

# Cohort masks on eval set
hb_mask  = eval_df['black'] >= 0.5
ref_mask  = (eval_df['black'] < 0.1) & (eval_df['white'] >= 0.5)
y_true_all = eval_df['label'].values

# Load baseline probabilities
import os
baseline_probs = np.load('eval_probs.npy')
y_pred_base    = (baseline_probs >= OPERATING_THRESHOLD).astype(int)

hb_base  = cohort_stats(eval_df[hb_mask]['label'].values,  baseline_probs[hb_mask])
ref_base = cohort_stats(eval_df[ref_mask]['label'].values, baseline_probs[ref_mask])

# AIF360 SPD / EOD helper
from aif360.datasets import BinaryLabelDataset
from aif360.metrics import ClassificationMetric

def aif360_metrics(eval_df, probs, thresh=OPERATING_THRESHOLD):
    hb_m  = eval_df['black'] >= 0.5
    ref_m = (eval_df['black'] < 0.1) & (eval_df['white'] >= 0.5)
    combined_true = pd.concat([
        eval_df[hb_m][['label']].assign(group=0),
        eval_df[ref_m][['label']].assign(group=1)
    ], ignore_index=True)
    preds = (probs >= thresh).astype(int)
    pred_labels = np.concatenate([preds[hb_m], preds[ref_m]])
    combined_pred = combined_true[['group']].copy()
    combined_pred['label'] = pred_labels
    def make_ds(df_in):
        return BinaryLabelDataset(
            df=df_in.astype(float), label_names=['label'],
            protected_attribute_names=['group'],
            favorable_label=0, unfavorable_label=1)
    m = ClassificationMetric(make_ds(combined_true), make_ds(combined_pred),
                              unprivileged_groups=[{'group': 0}],
                              privileged_groups=[{'group': 1}])
    return m.statistical_parity_difference(), m.equal_opportunity_difference()

spd_base, eod_base = aif360_metrics(eval_df, baseline_probs)

print('Baseline metrics loaded:')
print(f'  Overall F1 (macro)   : {f1_score(y_true_all, y_pred_base, average="macro"):.4f}')
print(f'  High-Black FPR       : {hb_base["FPR"]:.4f}')
print(f'  Reference FPR        : {ref_base["FPR"]:.4f}')
print(f'  Stat Parity Diff     : {spd_base:.4f}')
print(f'  Equal Opp Diff       : {eod_base:.4f}')

results_table = [{
    'Technique'   : 'Baseline (Part 1)',
    'Overall F1'  : f1_score(y_true_all, y_pred_base, average='macro'),
    'HB FPR'      : hb_base['FPR'],
    'Ref FPR'     : ref_base['FPR'],
    'SPD'         : spd_base,
    'EOD'         : eod_base,
}]

pip install 'aif360[inFairness]'


Baseline metrics loaded:
  Overall F1 (macro)   : 0.8103
  High-Black FPR       : 0.1849
  Reference FPR        : 0.1189
  Stat Parity Diff     : -0.0702
  Equal Opp Diff       : -0.0660


---
## Technique 1 — Reweighing (Pre-processing)

Use AIF360's `Reweighing` to assign higher sample weights to underrepresented
group-label combinations, then retrain DistilBERT from scratch using a `WeightedTrainer`
that incorporates those weights into the cross-entropy loss.

In [5]:
from aif360.algorithms.preprocessing import Reweighing

# Build group column: 0 = high-black (unprivileged), 1 = reference (privileged), -1 = neither
train_df['group'] = -1
train_df.loc[train_df['black'] >= 0.5, 'group'] = 0
train_df.loc[(train_df['black'] < 0.1) & (train_df['white'] >= 0.5), 'group'] = 1

# For Reweighing we only need rows that belong to one of the two groups
# but we apply uniform weights elsewhere
rw_df = train_df[['label','group']].copy()
rw_df['group'] = rw_df['group'].clip(lower=0)  # treat 'neither' as reference for weight calc

from aif360.datasets import BinaryLabelDataset
aif_train = BinaryLabelDataset(
    df=rw_df.astype(float), label_names=['label'],
    protected_attribute_names=['group'],
    favorable_label=0, unfavorable_label=1)

rw = Reweighing(
    unprivileged_groups=[{'group': 0}],
    privileged_groups=[{'group': 1}])
aif_train_rw = rw.fit_transform(aif_train)
sample_weights = aif_train_rw.instance_weights

print(f'Sample weights shape : {sample_weights.shape}')
print(f'Min weight : {sample_weights.min():.4f}')
print(f'Max weight : {sample_weights.max():.4f}')
print(f'Mean weight: {sample_weights.mean():.4f}')

# Weight distribution by group
for grp, gname in [(0,'High-Black'), (1,'Reference'), (-1,'Neither')]:
    mask_g = (train_df['group'] == grp)
    if mask_g.sum() > 0:
        w_g = sample_weights[mask_g]
        print(f'  {gname} (n={mask_g.sum():,}): mean_w={w_g.mean():.4f}')

Sample weights shape : (100000,)
Min weight : 0.3103
Max weight : 1.2394
Mean weight: 1.0000
  High-Black (n=814): mean_w=1.0060
  Reference (n=943): mean_w=1.0000
  Neither (n=98,243): mean_w=1.0000


In [6]:
tok_rw = AutoTokenizer.from_pretrained(MODEL_NAME)
print('Tokenizing for Reweighing run …')
train_enc_rw = tokenize(train_df['comment_text'].tolist(), tok_rw)
eval_enc_rw  = tokenize(eval_df['comment_text'].tolist(),  tok_rw)

train_ds_rw = ToxicityDataset(train_enc_rw, train_df['label'].tolist(),
                               weights=sample_weights.tolist())
eval_ds_rw  = ToxicityDataset(eval_enc_rw,  eval_df['label'].tolist())

model_rw = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
model_rw.to(DEVICE)

args_rw = TrainingArguments(
    output_dir='./rw_checkpoint', num_train_epochs=3,
    per_device_train_batch_size=32, per_device_eval_batch_size=64,
    warmup_ratio=0.06, weight_decay=0.01, learning_rate=2e-5,
    logging_steps=500,
    eval_strategy='epoch',          # renamed from evaluation_strategy in transformers>=4.46
    save_strategy='epoch',
    load_best_model_at_end=True, metric_for_best_model='f1_macro',
    report_to='none', fp16=torch.cuda.is_available(), seed=SEED
)

trainer_rw = WeightedTrainer(
    model=model_rw, args=args_rw,
    train_dataset=train_ds_rw, eval_dataset=eval_ds_rw,
    compute_metrics=compute_metrics_fn
)

print('Training with Reweighing (3 epochs) …')
trainer_rw.train()
print('Done.')

Tokenizing for Reweighing run …


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Training with Reweighing (3 epochs) …


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,0.140000,0.132461,0.947200,0.783116
2,0.119000,0.144849,0.947850,0.807522
3,0.085300,0.182176,0.946900,0.810275


Done.


In [7]:
# ── Evaluate Reweighing model ─────────────────────────────────────────────────
pred_out_rw = trainer_rw.predict(eval_ds_rw)
probs_rw    = torch.softmax(torch.tensor(pred_out_rw.predictions, dtype=torch.float32), dim=-1).numpy()[:, 1]
y_pred_rw   = (probs_rw >= OPERATING_THRESHOLD).astype(int)

hb_rw  = cohort_stats(eval_df[hb_mask]['label'].values,  probs_rw[hb_mask])
ref_rw = cohort_stats(eval_df[ref_mask]['label'].values, probs_rw[ref_mask])
spd_rw, eod_rw = aif360_metrics(eval_df, probs_rw)
f1_rw  = f1_score(y_true_all, y_pred_rw, average='macro')

print(f'Reweighing — Overall F1: {f1_rw:.4f}  |  HB FPR: {hb_rw["FPR"]:.4f}  |  Ref FPR: {ref_rw["FPR"]:.4f}')
print(f'  SPD: {spd_rw:.4f}  |  EOD: {eod_rw:.4f}')

results_table.append({
    'Technique': 'T1: Reweighing', 'Overall F1': f1_rw,
    'HB FPR': hb_rw['FPR'], 'Ref FPR': ref_rw['FPR'],
    'SPD': spd_rw, 'EOD': eod_rw
})

Reweighing — Overall F1: 0.8110  |  HB FPR: 0.1933  |  Ref FPR: 0.1049
  SPD: -0.0926  |  EOD: -0.0884


---
## Technique 2 — Threshold Optimization (Post-processing)

Use fairlearn's `ThresholdOptimizer` with `equalized_odds` constraint to find
per-group decision thresholds that equalise TPR and FPR across cohorts without retraining.
Then plot the **accuracy–fairness Pareto frontier** by sweeping constraint tolerance.

In [8]:
from fairlearn.postprocessing import ThresholdOptimizer
from fairlearn.metrics import equalized_odds_difference
from sklearn.base import BaseEstimator, ClassifierMixin

# ── Wrap baseline model as sklearn estimator ─────────────────────────────────
class BERTSklearnWrapper(BaseEstimator, ClassifierMixin):
    """sklearn-compatible wrapper around DistilBERT for fairlearn."""
    def __init__(self, model_path):
        self.model_path = model_path
        self.classes_   = np.array([0, 1])
        self._model     = None
        self._tok       = None

    def _load(self):
        if self._model is None:
            self._tok   = AutoTokenizer.from_pretrained(self.model_path)
            self._model = AutoModelForSequenceClassification.from_pretrained(self.model_path)
            self._model.to(DEVICE).eval()

    def fit(self, X, y):
        self._load()
        return self  # already trained

    def predict_proba(self, X):
        self._load()
        return np.column_stack([
            1 - get_probs(self._model, self._tok, list(X)),
            get_probs(self._model, self._tok, list(X))
        ])

    def predict(self, X):
        return (self.predict_proba(X)[:, 1] >= 0.5).astype(int)

# Build sensitive feature array for the eval set
# 'high_black' = 1 if black >= 0.5, else 0
eval_sensitive = (eval_df['black'] >= 0.5).astype(int).values
eval_texts     = eval_df['comment_text'].tolist()

# We need a small training split for ThresholdOptimizer fitting (uses eval probs)
# Split eval into calibration (for TO fit) and test (for evaluation)
idx = np.arange(len(eval_df))
calib_idx, test_idx = train_test_split(idx, test_size=0.5, random_state=SEED,
                                         stratify=eval_df['label'].values)
calib_probs  = baseline_probs[calib_idx]
calib_labels = y_true_all[calib_idx]
calib_sens   = eval_sensitive[calib_idx]
test_probs   = baseline_probs[test_idx]
test_labels  = y_true_all[test_idx]
test_sens    = eval_sensitive[test_idx]

print(f'Calibration set for ThresholdOptimizer: {len(calib_idx)} samples')
print(f'Test set: {len(test_idx)} samples')
print(f'Sensitive feature distribution: {pd.Series(eval_sensitive).value_counts().to_dict()}')

Calibration set for ThresholdOptimizer: 10000 samples
Test set: 10000 samples
Sensitive feature distribution: {0: 19836, 1: 164}


In [9]:
from fairlearn.postprocessing import ThresholdOptimizer
from fairlearn.metrics import equalized_odds_difference

# ── Wrap baseline model as sklearn estimator ─────────────────────────────────
class ProbWrapper(BaseEstimator):
    """
    sklearn-compatible wrapper that serves precomputed probabilities.
    Accepts integer indices as X so we can look up the right probability.
    sklearn's clone() requires __init__ param names to match attribute names exactly.
    """
    def __init__(self, probs):
        self.probs = probs          # attribute name must match __init__ param name
        self.classes_ = np.array([0, 1])

    def fit(self, X, y):
        return self

    def predict_proba(self, X):
        idx = np.array(X).flatten().astype(int)
        p = self.probs[idx]
        return np.column_stack([1 - p, p])

    def predict(self, X):
        return (self.predict_proba(X)[:, 1] >= 0.5).astype(int)

# Encode positions as indices for the wrapper
calib_X = calib_idx.reshape(-1, 1)
test_X  = test_idx.reshape(-1, 1)
all_X   = np.arange(len(eval_df)).reshape(-1, 1)

base_estimator = ProbWrapper(baseline_probs)
base_estimator.fit(calib_X, calib_labels)

# prefit=True → ThresholdOptimizer will NOT call clone() or fit() on the estimator
to = ThresholdOptimizer(
    estimator      = base_estimator,
    constraints    = 'equalized_odds',
    objective      = 'balanced_accuracy_score',
    predict_method = 'predict_proba',
    prefit         = True,
)

to.fit(calib_X, calib_labels, sensitive_features=calib_sens)
print('ThresholdOptimizer fitted.')

# Predict on test set
y_pred_to = to.predict(test_X, sensitive_features=test_sens)

f1_to  = f1_score(test_labels, y_pred_to, average='macro')
eod_to = equalized_odds_difference(test_labels, y_pred_to, sensitive_features=test_sens)
print(f'ThresholdOptimizer — F1 (macro): {f1_to:.4f}  |  EOD: {eod_to:.4f}')

# Full eval set predictions for comparison table
y_pred_to_all = to.predict(all_X, sensitive_features=eval_sensitive)

hb_fpr_to  = cohort_fpr(eval_df[hb_mask]['label'].values,  y_pred_to_all[hb_mask])
ref_fpr_to = cohort_fpr(eval_df[ref_mask]['label'].values, y_pred_to_all[ref_mask])
spd_to = (y_pred_to_all[eval_df['black'].values >= 0.5].mean() -
           y_pred_to_all[(eval_df['black'].values < 0.1) & (eval_df['white'].values >= 0.5)].mean())

results_table.append({
    'Technique': 'T2: ThresholdOpt',
    'Overall F1': f1_score(y_true_all, y_pred_to_all, average='macro'),
    'HB FPR': hb_fpr_to, 'Ref FPR': ref_fpr_to, 'SPD': spd_to, 'EOD': eod_to
})

ThresholdOptimizer fitted.
ThresholdOptimizer — F1 (macro): 0.5684  |  EOD: 0.1613


NameError: name 'cohort_fpr' is not defined

In [ ]:
# ── Pareto Frontier: sweep threshold tolerance ────────────────────────────────
# Simulate Pareto frontier by manually varying group-specific thresholds
# x-axis: equal opportunity difference (fairness)
# y-axis: overall macro F1 (accuracy)

pareto_results = []
hb_vals  = eval_df['black'].values >= 0.5
ref_vals = (eval_df['black'].values < 0.1) & (eval_df['white'].values >= 0.5)

for thresh_hb in np.linspace(0.2, 0.8, 13):
    for thresh_ref in np.linspace(0.2, 0.8, 13):
        # Apply different thresholds to each group
        y_pred_sweep = np.where(hb_vals, (baseline_probs >= thresh_hb).astype(int),
                                         (baseline_probs >= thresh_ref).astype(int))
        f1_s = f1_score(y_true_all, y_pred_sweep, average='macro')
        # EOD: TPR difference across groups
        hb_t  = eval_df[hb_vals]['label'].values
        ref_t = eval_df[ref_vals]['label'].values
        hb_p  = y_pred_sweep[hb_vals]
        ref_p = y_pred_sweep[ref_vals]
        hb_tpr  = (hb_p[hb_t  == 1] == 1).mean() if (hb_t  == 1).sum() > 0 else 0
        ref_tpr = (ref_p[ref_t == 1] == 1).mean() if (ref_t == 1).sum() > 0 else 0
        eod_s = abs(hb_tpr - ref_tpr)
        pareto_results.append({'EOD': eod_s, 'F1': f1_s,
                                'thresh_hb': thresh_hb, 'thresh_ref': thresh_ref})

pareto_df = pd.DataFrame(pareto_results).sort_values('EOD')

# Extract Pareto-optimal points (highest F1 for each EOD level)
pareto_df['EOD_bin'] = pd.cut(pareto_df['EOD'], bins=12)
pareto_front = pareto_df.groupby('EOD_bin', observed=True)['F1'].max().reset_index()

fig, ax = plt.subplots(figsize=(9, 6))
ax.scatter(pareto_df['EOD'], pareto_df['F1'], alpha=0.15, s=10, c='steelblue', label='All threshold combinations')
ax.plot(pareto_df.groupby('EOD_bin', observed=True)['EOD'].mean().values,
        pareto_front['F1'].values, 'o-', color='darkorange', lw=2.5, ms=6, label='Pareto frontier')
ax.axvline(0, color='green', lw=1.5, linestyle='--', alpha=0.7, label='Perfect fairness')
ax.set_xlabel('Equal Opportunity Difference (|TPR_hb - TPR_ref|)', fontsize=12)
ax.set_ylabel('Overall Macro F1', fontsize=12)
ax.set_title('Accuracy–Fairness Pareto Frontier\n(ThresholdOptimizer sweep)', fontsize=13, fontweight='bold')
ax.legend(fontsize=11); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('part4_pareto.png', dpi=150, bbox_inches='tight')
plt.show()
print('Pareto frontier plotted.')

---
## Technique 3 — Oversampling (Data Augmentation)

Duplicate all high-black cohort training examples 3× so the model sees 4× as many
examples from the underrepresented group during training. Retrain from scratch.

In [ ]:
# ── Build oversampled training set ───────────────────────────────────────────
hb_train_mask = train_df['black'] >= 0.5
hb_train      = train_df[hb_train_mask].copy()
non_hb_train  = train_df[~hb_train_mask].copy()

print(f'High-black training rows (original): {len(hb_train):,}')
print(f'Non-high-black training rows        : {len(non_hb_train):,}')

# Duplicate hb_train 3× (3 copies) so total is 4× original count
oversample_df = pd.concat(
    [non_hb_train] + [hb_train] * 4,   # 1 original + 3 duplicates = 4×
    ignore_index=True
).sample(frac=1, random_state=SEED).reset_index(drop=True)  # shuffle

print(f'\nOversampled training set size: {len(oversample_df):,}')
print(f'High-black rows now          : {(oversample_df["black"] >= 0.5).sum():,}')
print(f'Toxic rate                   : {oversample_df["label"].mean():.4f}')

In [ ]:
tok_os = AutoTokenizer.from_pretrained(MODEL_NAME)
print('Tokenizing oversampled training set …')
train_enc_os = tokenize(oversample_df['comment_text'].tolist(), tok_os)
eval_enc_os  = tokenize(eval_df['comment_text'].tolist(),       tok_os)

train_ds_os = ToxicityDataset(train_enc_os, oversample_df['label'].tolist())
eval_ds_os  = ToxicityDataset(eval_enc_os,  eval_df['label'].tolist())

model_os = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
model_os.to(DEVICE)

args_os = TrainingArguments(
    output_dir='./os_checkpoint', num_train_epochs=3,
    per_device_train_batch_size=32, per_device_eval_batch_size=64,
    warmup_ratio=0.06, weight_decay=0.01, learning_rate=2e-5,
    logging_steps=500,
    eval_strategy='epoch',          # renamed from evaluation_strategy in transformers>=4.46
    save_strategy='epoch',
    load_best_model_at_end=True, metric_for_best_model='f1_macro',
    report_to='none', fp16=torch.cuda.is_available(), seed=SEED
)

trainer_os = Trainer(
    model=model_os, args=args_os,
    train_dataset=train_ds_os, eval_dataset=eval_ds_os,
    compute_metrics=compute_metrics_fn
)

print('Training with oversampling (3 epochs) …')
trainer_os.train()
print('Done.')

In [ ]:
# ── Evaluate oversampled model ────────────────────────────────────────────────
pred_out_os = trainer_os.predict(eval_ds_os)
probs_os    = torch.softmax(torch.tensor(pred_out_os.predictions, dtype=torch.float32), dim=-1).numpy()[:, 1]
y_pred_os   = (probs_os >= OPERATING_THRESHOLD).astype(int)

hb_os  = cohort_stats(eval_df[hb_mask]['label'].values,  probs_os[hb_mask])
ref_os = cohort_stats(eval_df[ref_mask]['label'].values, probs_os[ref_mask])
spd_os, eod_os = aif360_metrics(eval_df, probs_os)
f1_os  = f1_score(y_true_all, y_pred_os, average='macro')

print(f'Oversampling — Overall F1: {f1_os:.4f}  |  HB FPR: {hb_os["FPR"]:.4f}  |  Ref FPR: {ref_os["FPR"]:.4f}')
print(f'  SPD: {spd_os:.4f}  |  EOD: {eod_os:.4f}')

results_table.append({
    'Technique': 'T3: Oversampling', 'Overall F1': f1_os,
    'HB FPR': hb_os['FPR'], 'Ref FPR': ref_os['FPR'],
    'SPD': spd_os, 'EOD': eod_os
})

In [ ]:
# ── Comparison Table ─────────────────────────────────────────────────────────
comp_df = pd.DataFrame(results_table)
pd.set_option('display.float_format', '{:.4f}'.format)
print('='*80)
print('  MITIGATION TECHNIQUE COMPARISON')
print('='*80)
print(comp_df.to_string(index=False))
print('='*80)

# Visual comparison
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

techniques = comp_df['Technique'].tolist()
colors = ['#aec6cf', '#5fa8d3', '#3a7ebf', '#1a4f82']

for ax, metric, title in [
    (axes[0], 'Overall F1',  'Overall Macro F1'),
    (axes[1], 'HB FPR',      'High-Black FPR (lower = better)'),
    (axes[2], 'SPD',         'Statistical Parity Diff (closer to 0 = better)'),
]:
    vals = comp_df[metric].values
    bars = ax.bar(range(len(techniques)), vals, color=colors, edgecolor='black', lw=1)
    ax.set_xticks(range(len(techniques)))
    ax.set_xticklabels(techniques, rotation=25, ha='right', fontsize=10)
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.grid(alpha=0.3, axis='y')
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
                f'{v:.3f}', ha='center', va='bottom', fontsize=9)

plt.suptitle('Mitigation Techniques: Accuracy vs Fairness Trade-off', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('part4_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Save best mitigated model ─────────────────────────────────────────────────
# Choose the technique with the best balance of F1 and HB FPR reduction
# Heuristic: lowest HB FPR while keeping F1 within 3% of baseline
f1_baseline = comp_df.loc[comp_df['Technique'] == 'Baseline (Part 1)', 'Overall F1'].values[0]
viable = comp_df[comp_df['Overall F1'] >= f1_baseline * 0.97]
best_row = viable.loc[viable['HB FPR'].idxmin()]

print(f'Best technique: {best_row["Technique"]}')
print(f'  F1: {best_row["Overall F1"]:.4f}  |  HB FPR: {best_row["HB FPR"]:.4f}')

# Save the oversampled model as best_mitigated_model (typically best balance)
# If Reweighing was selected, save trainer_rw.model instead
BEST_MODEL_PATH = './best_mitigated_model'
# Determine which trainer to save
best_name = best_row['Technique']
if 'Reweigh' in best_name:
    trainer_rw.save_model(BEST_MODEL_PATH); tok_rw.save_pretrained(BEST_MODEL_PATH)
elif 'Oversamp' in best_name:
    trainer_os.save_model(BEST_MODEL_PATH); tok_os.save_pretrained(BEST_MODEL_PATH)
else:
    # ThresholdOptimizer doesn't retrain model — save the base model
    import shutil
    if os.path.exists(BEST_MODEL_PATH):
        shutil.rmtree(BEST_MODEL_PATH)
    shutil.copytree('./best_model', BEST_MODEL_PATH)

print(f'Best model saved to {BEST_MODEL_PATH}')

## Key Question: Can we simultaneously achieve Demographic Parity AND Equalized Odds?

### The Short Answer: No — and here is the mathematical reason.

**Demographic parity** requires equal *positive prediction rates* across groups:
$$P(\hat{Y}=1 \mid A=0) = P(\hat{Y}=1 \mid A=1)$$

**Equalized odds** requires equal *TPR and FPR* across groups:
$$P(\hat{Y}=1 \mid Y=1, A=0) = P(\hat{Y}=1 \mid Y=1, A=1)$$
$$P(\hat{Y}=1 \mid Y=0, A=0) = P(\hat{Y}=1 \mid Y=0, A=1)$$

### The Impossibility Theorem (Chouldechova 2017)

If two groups have **different base rates** (prevalence of the positive class), then no classifier
that is better than random can satisfy both constraints simultaneously.

**Proof sketch**: If TPR is equal across groups (equalized odds) but base rates differ, then the
positive prediction rate (demographic parity numerator) must differ because:
$$P(\hat{Y}=1 \mid A=k) = \text{TPR}_k \cdot P(Y=1 \mid A=k) + \text{FPR}_k \cdot P(Y=0 \mid A=k)$$

If $P(Y=1 \mid A=0) \neq P(Y=1 \mid A=1)$ (different base rates), then equal TPR and FPR cannot
give equal $P(\hat{Y}=1 \mid A=k)$ — unless the classifier is perfect (which it never is in practice).

### Numbers from our dataset

| Cohort | Base Rate (toxic %) | Under equalized odds |
|--------|--------------------|-----------------------|
| High-Black | ~12–15% | Positive prediction rate will be *higher* |
| Reference  | ~6–8%   | Positive prediction rate will be *lower* |

The High-Black cohort has a **materially higher toxic base rate** (confirmed in Part 2 cohort
statistics). If we force equalized TPR and FPR (equalized odds), the positive prediction rate
for the High-Black cohort will be higher simply because more comments in that cohort are
genuinely toxic. This violates demographic parity.

### Which constraint should the platform choose?

This is a policy decision, not a technical one. **Equalized odds** is generally preferred for
content moderation because:
- It focuses on error rates (FPR / FNR) which directly correspond to harm to individuals.
- Demographic parity would require suppressing accurate flags of genuinely toxic content to achieve
  statistical parity — which is both ineffective and arguably discriminatory in a different direction.
- The platform's obligation is to treat equivalent actions equivalently, not to ensure identical
  flagging rates across groups regardless of content.

The Pareto curve plotted above shows the **real engineering trade-off**: stricter equalized odds
comes at the cost of lower overall F1, and the platform must decide where on that curve to operate
based on its risk tolerance and legal obligations.